---
technologies: "Data preprocess with Local-Cloud integration"
category: "Demonstration of technology"
difficulty: "Expert"
---

# Data Preprocess with Local-Cloud Integration

## Used material

1. <span id="used-material-1"></span> [ICEbreaker package](https://github.com/K123AsJ0k1/ice-mlops-mlch/tree/main/applications/packages/icebreaker)

## How to do data preprocess with Local-Cloud integration?

Assuming you have done feature extraction on the data using the [data analysis chapter](./13_data_analysis_with_local_cloud_integration.ipynb), we can proceed with the demonstrations data preprocessing. The objectives of this stage are the following:

- Create a validation dataset by sampling from internal material
- Create a general dataset by sampling from external material
- Put all the internal material into RAG database
- Create a synthetic dataset by using internal material and asking a model to create factual, synthesis and negative QA pairs

We will first do local sampling and then run a local-cloud pipeline to utilize available GPUs for the RAG setup and dataset generaton:

- Local
    - Select 150 QA pairs from extracted internal material for validation dataset
    - Select 100 QA pairs from extracted external material for general dataset
- Local-cloud
    - Process around 447+ chunks into Qdrant database for hybrid search using Alibaba-NLP/gte-modernbert-base and opensearch-project/opensearch-neural-sparse-encoding-doc-v3-distill
    - Generate 120 QA pair synthetic dataset from the validation dataset with factual, synthesis and negative types using deepseek-R1
    - Used the 447+ chunks to query the Qdrant database to get RAG metrics for ideal dataset

Everything will be done with the help of the ICEbreaker package [(1)](#used-material-1). We will begin with setting up the enviroment:

In [ ]:
from icebreaker.workflow.utility import workflow_setup_enviroment
from icebreaker.mlflow.setup import mlflow_setup_client

workflow_setup = workflow_setup_enviroment(
    env_path = '(fill)/.ssh/.env',
    swift_auth_url = 'https://pouta.csc.fi:5001/v3',
    swift_auth_version = '3',
    swift_pre_auth_url = 'https://a3s.fi:443/swift/v1'
)

workflow_mlflow_client = mlflow_setup_client(
    mlflow_parameters = {
        'tracking-uri': 'http://mlflow.oss:7001',
        's3-endpoint-url': 'http://localhost:7009',
        'aws-access-key-id': 'minioadmin',
        'aws-secret-access-key': 'minioadmin'
    }
)

We can create the validation dataset in the following way:

In [ ]:
from icebreaker.workflow.utility import workflow_create_validation_dataset

validation_dataset = workflow_create_validation_dataset(
    swift_client = workflow_setup['swift-client'],
    mlflow_client = work_mlflow_client,
    dataset_paths = [
        'DATA/EXTRACTED/extraction-data-internal-tutorial-v095-part-1.pkl',
        'DATA/EXTRACTED/extraction-data-internal-tutorial-v095-part-2.pkl',
        'DATA/EXTRACTED/extraction-data-internal-tutorial-v095-part-3.pkl',
        'DATA/EXTRACTED/extraction-data-internal-tutorial-v095-part-4.pkl',
        'DATA/EXTRACTED/extraction-data-internal-tutorial-v095-part-5.pkl',
        'DATA/EXTRACTED/extraction-data-internal-tutorial-v095-part-6.pkl',
        'DATA/EXTRACTED/extraction-data-internal-tutorial-v095-part-7.pkl',
        'DATA/EXTRACTED/extraction-data-internal-tutorial-v095-part-8.pkl',
    ],
    storage_parameters = {
        'bucket-target': 'experiment',
        'bucket-prefix': '(fill unique prefix)',
        'bucket-user': 'user@example.com',
        'object-serialization': 'pickle'
    },
    experiment_name = 'mlch-code-assistant-rag-pipeline',
    dataset_tags = {
        'workflow-type': 'local-cloud-hpc',
        'pipeline-step': 'validation-dataset-sampling',
    },
    dataset_user = 'user'
)

We can create the general dataset the following way:

In [ ]:
from icebreaker.workflow.utility import workflow_create_general_dataset
from icebreaker.misc.yaml import get_validated_yamls

external_dataset_sampling = get_validated_yamls(
    relative_paths = [
        'external-dataset-sampling.yaml',
    ],
    yaml_validator = None,
    key_replacer = {}
)

general_dataset = workflow_create_general_dataset(
    swift_client = workflow_setup['swift-client'],
    mlflow_client = work_mlflow_client,
    sampling_parameters = external_dataset_sampling['external-dataset-sampling'],
    storage_parameters = {
        'bucket-target': 'experiment',
        'bucket-prefix': '(fill unique prefix)',
        'bucket-user': 'user@example.com',
        'object-serialization': 'parquet'
    },
    dataset_paths = [
        'DATA/EXTRACTED/extraction-data-external-bash-command-data.parquet',
        'DATA/EXTRACTED/extraction-data-external-openai-humaneval.parquet',
        'DATA/EXTRACTED/extraction-data-external-mbpp.parquet',
        'DATA/EXTRACTED/extraction-data-external-opc-sft-stage2-educational-instruct-1.parquet',
        'DATA/EXTRACTED/extraction-data-external-opc-sft-stage2-package-instruct-1.parquet',
        'DATA/EXTRACTED/extraction-data-external-opc-sft-stage2-package-instruct-2.parquet',
        'DATA/EXTRACTED/extraction-data-external-opc-sft-stage2-package-instruct-3.parquet',
        'DATA/EXTRACTED/extraction-data-external-opc-sft-stage2-mceval-instruct-1.parquet',
        'DATA/EXTRACTED/extraction-data-external-opc-sft-stage1-largescale-diverse-instruct-1.parquet',
        'DATA/EXTRACTED/extraction-data-external-opc-sft-stage1-largescale-diverse-instruct-2.parquet',
        'DATA/EXTRACTED/extraction-data-external-opc-sft-stage1-largescale-diverse-instruct-3.parquet',
        'DATA/EXTRACTED/extraction-data-external-opc-sft-stage2-evol-instruct-1.parquet',
        'DATA/EXTRACTED/extraction-data-external-opc-sft-stage2-evol-instruct-2.parquet',
        'DATA/EXTRACTED/extraction-data-external-opc-sft-stage2-evol-instruct-3.parquet',
        'DATA/EXTRACTED/extraction-data-external-opc-sft-stage1-realuser-instruct-1.parquet',
        'DATA/EXTRACTED/extraction-data-external-opc-sft-stage1-realuser-instruct-2.parquet',
        'DATA/EXTRACTED/extraction-data-external-opc-sft-stage1-realuser-instruct-3.parquet',
        'DATA/EXTRACTED/extraction-data-external-opc-sft-stage1-filtered-infinity-instruct-1.parquet',
        'DATA/EXTRACTED/extraction-data-external-opc-sft-stage1-filtered-infinity-instruct-2.parquet',
        'DATA/EXTRACTED/extraction-data-external-opc-sft-stage1-filtered-infinity-instruct-3.parquet'
    ],
    experiment_name = 'mlch-code-assistant-rag-pipeline',
    dataset_tags = {
        'workflow-type': 'local-cloud-hpc',
        'pipeline-step': 'validation-dataset-sampling',
    },
    dataset_user = 'user'
)

With the local sampling done, we can setup a loca-cloud data preprocess pipeline using the [local parameters](./parameters/local-cluster-parameters.yaml), [cloud parameters](./parameters/cloud-cluster-parameters.yaml), [rag setup Ray script](./ray/data_preprocess_rag_setup/main.py), [data generator Ray script](./ray/data_preprocess_dataset_generation/main.py), [rag evalution Ray script](./ray/data_preprocess_rag_evaluation/main.py) and [pipeline details](./details/local_cloud_data_preprocess_details.yaml) in the following way:

In [ ]:
from icebreaker.models.clusters import Clusters
from icebreaker.misc.yaml import get_validated_yamls
from icebreaker.misc.dict import update_nested_dict
from icebreaker.costs.use import costs_local_spending_per_second, costs_csc_cloud_spending_per_second
from icebreaker.objects.use import objects_nested_update
from icebreaker.ray.setup import ray_store_job
from icebreaker.setup.fragments import compose_yaml_dict
from icebreaker.setup.pipeline import create_pipeline_inputs

local_cloud_cluster_yamls = get_validated_yamls(
    relative_paths = [
        './parameters/local-cluster-parameters.yaml',
        './parameters/cloud-cluster-parameters.yaml'
    ],
    yaml_validator = Clusters,
    key_replacer = {
        'local-cluster-parameters': 'local',
        'cloud-cluster-parameters': 'cloud'
    }
)

controller_update_dict = {
    'local': {
        'clusters': {
            'lt1': {
                'activate': False
            },
            'lt2': {
                'activate': True
            },
            'lt3': {
                'activate': True
            }
        }
    },
    'cloud': {
        'clusters': {
            'vm2': {
                'activate': False
            }
        }
    }
}

pipeline_cluster_yaml = update_nested_dict(
    target_dict = local_cloud_cluster_yamls, 
    update_dict = controller_update_dict
)

cost_update_dict = {
    'local': {
        'clusters': {
            'lt1': {
                'utilization-cost-sec': costs_local_spending_per_second(
                    hardware_cost = local_cloud_cluster_yamls['local']['clusters']['lt1']['hardware-costs'],
                    expected_years = local_cloud_cluster_yamls['local']['clusters']['lifetime-expecation-year'],
                )
            },
            'lt2': {
                'utilization-cost-sec': costs_local_spending_per_second(
                    hardware_cost = local_cloud_cluster_yamls['local']['clusters']['lt2']['hardware-costs'],
                    expected_years = local_cloud_cluster_yamls['local']['clusters']['lifetime-expecation-year'],
                )
            },
            'lt3': {
                'utilization-cost-sec': costs_local_spending_per_second(
                    hardware_cost = local_cloud_cluster_yamls['local']['clusters']['lt3']['hardware-costs'],
                    expected_years = local_cloud_cluster_yamls['local']['clusters']['lifetime-expecation-year'],
                )
            }
        }
    },
    'cloud': {
        'clusters': {
            'vm1': {
                'utilization-cost-sec': costs_csc_cloud_spending_per_second(
                    package_cost_euros = local_cloud_cluster_yamls['cloud']['clusters']['package-cost-euros'],
                    package_billing_units = local_cloud_cluster_yamls['cloud']['clusters']['package-billing-units'],
                    spent_units_per_hour = local_cloud_cluster_yamls['cloud']['clusters']['vm1']['spent-units-per-hour']
                )
            },
            'vm2': {
                'utilization-cost-sec': costs_csc_cloud_spending_per_second(
                    package_cost_euros = local_cloud_cluster_yamls['cloud']['clusters']['package-cost-euros'],
                    package_billing_units = local_cloud_cluster_yamls['cloud']['clusters']['package-billing-units'],
                    spent_units_per_hour = local_cloud_cluster_yamls['cloud']['clusters']['vm2']['spent-units-per-hour']
                )
            }
        }
    }
}

pipeline_cluster_yaml = update_nested_dict(
    target_dict = pipeline_cluster_yaml, 
    update_dict = cost_update_dict
)

update_status = objects_nested_update(
    swift_client = workflow_setup['swift-client'],
    storage_parameters = {
        'bucket-target': 'pipeline',
        'bucket-prefix': '(fill unique prefix)',
        'bucket-user': 'user@example.com',
        'object-name': 'mana',
        'object-serialization': 'pickle',
        'path-replacers': {
            'name': 'local-cloud-cluster.pkl'
        },
        'path-names': [],
        'debug-prints': True,
        'lock-parameters': {},
        'lock-location': None,
        'overwrite': True
    },
    object_input = controller_update_dict
)

used_job_storing_time_1 = ray_store_job(
    storage_client = workflow_setup['swift-client'],
    storage_parameters = {
        'bucket-target': 'pipeline',
        'bucket-prefix': '(fill unique prefix)',
        'bucket-user': 'user@example.com'
    },
    working_directory_path = '(fill)/ice-mlops-mlch/tutorials/studying/part-7/ray/data_preprocess_rag_setup'
)

used_job_storing_time_2 = ray_store_job(
    storage_client = workflow_setup['swift-client'],
    storage_parameters = {
        'bucket-target': 'pipeline',
        'bucket-prefix': '(fill unique prefix)',
        'bucket-user': 'user@example.com'
    },
    working_directory_path = '(fill)/ice-mlops-mlch/tutorials/studying/part-7/ray/data_preprocess_dataset_generation'
)

used_job_storing_time_3 = ray_store_job(
    storage_client = workflow_setup['swift-client'],
    storage_parameters = {
        'bucket-target': 'pipeline',
        'bucket-prefix': '(fill unique prefix)',
        'bucket-user': 'user@example.com'
    },
    working_directory_path = '(fill)/ice-mlops-mlch/tutorials/studying/part-7/ray/data_preprocess_rag_evaluation'
)

workflow_steps = [
    'step-1',
    'step-2',
    'step-3'
]

time_object_name_update = {
    'storage': 'fill',
    'processing': {
        'step-1': {
            'general': 'fill'
        },
        'step-2': {
            'general': 'fill'
        },
        'step-3': {
            'general': 'fill'
        }
    }
}

data_preprocess_pipeline_arguments = compose_yaml_dict(
    fragments_folder = '(fill)/ice-mlops-mlch/tutorials/studying/part-7/fragments',
    fragments_prefix = '_fragment.yaml',
    root_fragment = 'parameters',
    details_folder = '(fill)ice-mlops-mlch/tutorials/studying/part-7/details',
    details_prefix = '_preprocess_details.yaml',
    env_dict = {},
    dict_inputs = {
        'run': 'local-cloud-data-preprocess-run',
        'experiment': 'local-cloud-rag-assistant-experiment',
        'storage|swift': workflow_setup['swift-parameters'],
        'integration|cluster-yamls': pipeline_cluster_yaml,
        'integration|workflow-steps': workflow_steps,
        'processing|step-1|general|swift': workflow_setup['swift-parameters'],
        'processing|step-1|cluster|cloud-vm1|job|runtime|env_vars|HF_TOKEN': workflow_setup['hf-token'],
        'processing|step-1|cluster|local-lt2|job|runtime|env_vars|HF_TOKEN': workflow_setup['hf-token'],
        'processing|step-1|cluster|local-lt3|job|runtime|env_vars|HF_TOKEN': workflow_setup['hf-token'],
        'processing|step-2|general|swift': workflow_setup['swift-parameters'],
        'processing|step-2|cluster|cloud-vm1|mlflow|trace-attributes|utilization-cost-sec': pipeline_cluster_yaml['cloud']['clusters']['vm1']['utilization-cost-sec'],
        'processing|step-2|cluster|cloud-vm1|mlflow|trace-attributes|utilization-cost-currency': pipeline_cluster_yaml['cloud']['clusters']['vm1']['utilization-cost-currency'],
        'processing|step-2|cluster|cloud-vm1|job|runtime|env_vars|HF_TOKEN': workflow_setup['hf-token'],
        'processing|step-2|cluster|local-lt2|mlflow|trace-attributes|utilization-cost-sec': pipeline_cluster_yaml['local']['clusters']['lt2']['utilization-cost-sec'],
        'processing|step-2|cluster|local-lt2|mlflow|trace-attributes|utilization-cost-currency': pipeline_cluster_yaml['local']['clusters']['lt2']['utilization-cost-currency'],
        'processing|step-2|cluster|local-lt2|job|runtime|env_vars|HF_TOKEN': workflow_setup['hf-token'],
        'processing|step-2|cluster|local-lt3|mlflow|trace-attributes|utilization-cost-sec': pipeline_cluster_yaml['local']['clusters']['lt3']['utilization-cost-sec'],
        'processing|step-2|cluster|local-lt3|mlflow|trace-attributes|utilization-cost-currency': pipeline_cluster_yaml['local']['clusters']['lt3']['utilization-cost-currency'],
        'processing|step-2|cluster|local-lt3|job|runtime|env_vars|HF_TOKEN': workflow_setup['hf-token'],
        'processing|step-3|general|swift': workflow_setup['swift-parameters'],
        'processing|step-3|cluster|cloud-vm1|process|utilization-cost-sec': pipeline_cluster_yaml['cloud']['clusters']['vm1']['utilization-cost-sec'],
        'processing|step-3|cluster|cloud-vm1|process|utilization-cost-currency': pipeline_cluster_yaml['cloud']['clusters']['vm1']['utilization-cost-currency'],
        'processing|step-3|cluster|cloud-vm1|job|runtime|env_vars|HF_TOKEN': workflow_setup['hf-token']
    }, 
    output_folder = '(fill)/ice-mlops-mlch/tutorials/studying/part-7/generated',
    output_name = 'data_preprocess_pipeline_input'
)

from icebreaker.models.pipeline_arguments import PipelineArguments
pydantic_validation = PipelineArguments.model_validate(data_preprocess_pipeline_arguments)
pipeline_run_name = data_preprocess_pipeline_arguments ['run']
pipeline_experiment_name = data_preprocess_pipeline_arguments ['experiment']
if 'run' in data_preprocess_pipeline_arguments:
    del data_preprocess_pipeline_arguments['run']
if 'experiment' in data_preprocess_pipeline_arguments:
    del data_preprocess_pipeline_arguments['experiment']

Now, assuming you have access to KFP at http://kubeflow.oss:7001, we can setup a client, initilize the pipeline and run it in the following way:

In [ ]:
import kfp
KFP_ENDPOINT = "http://kubeflow.oss:7001"
kfp_client = kfp.Client(host=KFP_ENDPOINT)

In [ ]:
%run (fill)/ice-mlops-mlch/tutorials/studying/part-7/kubeflow/local_cloud_parallel_pipeline.py

In [ ]:
from icebreaker.kubeflow.use import kubeflow_manage_run
data_analysis_pipeline_time = kubeflow_manage_run(
    storage_client = workflow_setup['swift-client'],
    kfp_client = kfp_client,
    pipeline_function = local_cloud_parallel_pipeline,
    run_name = pipeline_run_name,
    experiment_name = pipeline_experiment_name,
    pipeline_arguments = data_preprocess_pipeline_arguments,
    cache_steps = False,
    time_name = 'kubeflow-pipeline-local-lt1',
    time_update_object = time_object_name_update,
    loop_amount = 20000,
    loop_wait = 10
)

When the pipeline has completed, we can sample the generated datasets in the following way:

In [ ]:
from icebreaker.workflow.utility import workflow_create_synthetic_dataset

dataset_results = workflow_create_synthetic_dataset(
    swift_client = workflow_setup['swift-client'],
    mlflow_client = work_mlflow_client,
    storage_parameters = {
        'bucket-target': 'experiment',
        'bucket-prefix': '(fill unique prefix)',
        'bucket-user': 'user@example.com',
        'object-serialization': 'parquet'
    },
    dataset_paths = [
        'DATA/GENERATED/preprocess-generator-data-cloud-vm1-internal-validation-dataset-run-1.parquet',
        'DATA/GENERATED/preprocess-generator-data-local-lt2-internal-validation-dataset-run-1.parquet',
        'DATA/GENERATED/preprocess-generator-data-local-lt3-internal-validation-dataset-run-1.parquet',
    ],
    sample_amount = 120,
    experiment_name = 'mlch-code-assistant-rag-pipeline',
    dataset_tags = {
        'workflow-type': 'local-cloud-hpc',
        'pipeline-step': 'synthetic-dataset-sampling',
    },
    dataset_user = 'user'
)

Finally, we can get the times and costs of each pipeline step across infrastructure in the following way:

In [ ]:
from icebreaker.objects.use import objects_get_data
pipeline_time_object = objects_get_data(
    swift_client = workflow_setup['swift-client'],
    storage_parameters = {
        'bucket-target': 'experiment',
        'bucket-prefix': '(fill unique prefix)',
        'bucket-user': 'user@example.com',
        'object-name': 'root',
        'object-serialization': 'parquet',
        'path-replacers': {
            'name': 'TIMES/local-cloud-workflow-2.parquet'
        },
        'path-names': [],
        'debug-prints': True,
        'lock-parameters': {},
        'lock-location': None,
        'overwrite': True
    },
    dict_format = False
)

In [6]:
from icebreaker.workflow.utility import workflow_time_stats

time_stats = workflow_time_stats(
    time_df = pipeline_time_object[0],
    cluster_yaml = pipeline_cluster_yaml
) 

In [7]:
print(f'Total pipeline cost: {time_stats['metrics']['total-pipeline-cost-eur']}')

Total pipeline cost: 1.1520276494246002


In [8]:
time_stats['tables'][0]

,name,start,end,time,infrastructure,stage,cost
0,kubeflow-pipeline-local-lt1,1.793797e+09,1.793800e+09,3171.59834,local-lt1,step,0.055281
1,global-distribution-step-cloud-vm1,1.793797e+09,1.793797e+09,2.53673,cloud-vm1,step,0.000888
2,ray-internal-rag-setup-cloud-vm1-step-1,1.793798e+09,1.793798e+09,22.79466,cloud-vm1,step-1,0.007978
3,ray-internal-rag-setup-local-lt2-step-1,1.793798e+09,1.793798e+09,22.98824,local-lt2,step-1,0.000463
4,ray-internal-rag-setup-local-lt3-step-1,1.793798e+09,1.793798e+09,32.65763,local-lt3,step-1,0.001496
5,ray-internal-dataset-creation-cloud-vm1-step-2,1.793798e+09,1.793799e+09,554.27367,cloud-vm1,step-2,0.193996
6,ray-internal-rag-evalution-cloud-vm1-step-3,1.793799e+09,1.793799e+09,279.23551,cloud-vm1,step-3,0.097732
7,cluster-orhestractor-step-cloud-vm1-cluster-cl...,1.793798e+09,1.793799e+09,1496.20232,cloud-vm1,step,0.523671
8,ray-internal-dataset-creation-local-lt2-step-2,1.793798e+09,1.793800e+09,1910.83552,local-lt2,step-2,0.038446
9,cluster-orhestractor-step-cloud-vm1-cluster-lo...,1.793798e+09,1.793800e+09,2336.87227,cloud-vm1,step,0.817905


In [9]:
time_stats['tables'][1]

time                                                   \
                       sum        mean          std      median         min   
infrastructure                                                                
cloud-vm1       7426.97879  1060.99697  1131.804036   554.27367     2.53673   
local-lt1       3171.59834  3171.59834          NaN  3171.59834  3171.59834   
local-lt2       1933.82376   966.91188  1334.909614   966.91188    22.98824   
local-lt3       2175.90112  1087.95056  1492.409574  1087.95056    32.65763   

                                cost                                          \
                       max       sum      mean       std    median       min   
infrastructure                                                                 
cloud-vm1       2735.06363  2.599443  0.371349  0.396131  0.193996  0.000888   
local-lt1       3171.59834  0.055281  0.055281       NaN  0.055281  0.055281   
local-lt2       1910.83552  0.038909  0.019454  0.026858  0.019454  0.000463   
local-lt3       2143.24349  0.099678  0.049839  0.068367  0.049839  0.001496   

                          
                     max  
infrastructure            
cloud-vm1       0.957272  
local-lt1       0.055281  
local-lt2       0.038446  
local-lt3       0.098182

In [10]:
time_stats['tables'][2]

time                                                   \
               sum         mean          std      median        min   
stage                                                                 
step    9742.27329  1948.454658  1250.682928  2336.87227    2.53673   
step-1    78.44053    26.146843     5.639337    22.98824   22.79466   
step-2  4608.35268  1536.117560   858.205357  1910.83552  554.27367   
step-3   279.23551   279.235510          NaN   279.23551  279.23551   

                        cost                                                    
               max       sum      mean       std    median       min       max  
stage                                                                           
step    3171.59834  2.355017  0.471003  0.433995  0.523671  0.000888  0.957272  
step-1    32.65763  0.009937  0.003312  0.004074  0.001496  0.000463  0.007978  
step-2  2143.24349  0.330624  0.110208  0.078469  0.098182  0.038446  0.193996  
step-3   279.23551  0.097732  0.097732       NaN  0.097732  0.097732  0.097732

In [11]:
time_stats['tables'][3]

time                                         \
                              sum         mean          std       median   
infrastructure stage                                                       
cloud-vm1      step    6570.67495  1642.668738  1209.232616  1916.537295   
               step-1    22.79466    22.794660          NaN    22.794660   
               step-2   554.27367   554.273670          NaN   554.273670   
               step-3   279.23551   279.235510          NaN   279.235510   
local-lt1      step    3171.59834  3171.598340          NaN  3171.598340   
local-lt2      step-1    22.98824    22.988240          NaN    22.988240   
               step-2  1910.83552  1910.835520          NaN  1910.835520   
local-lt3      step-1    32.65763    32.657630          NaN    32.657630   
               step-2  2143.24349  2143.243490          NaN  2143.243490   

                                                   cost                      \
                              min         max       sum      mean       std   
infrastructure stage                                                          
cloud-vm1      step       2.53673  2735.06363  2.299736  0.574934  0.423231   
               step-1    22.79466    22.79466  0.007978  0.007978       NaN   
               step-2   554.27367   554.27367  0.193996  0.193996       NaN   
               step-3   279.23551   279.23551  0.097732  0.097732       NaN   
local-lt1      step    3171.59834  3171.59834  0.055281  0.055281       NaN   
local-lt2      step-1    22.98824    22.98824  0.000463  0.000463       NaN   
               step-2  1910.83552  1910.83552  0.038446  0.038446       NaN   
local-lt3      step-1    32.65763    32.65763  0.001496  0.001496       NaN   
               step-2  2143.24349  2143.24349  0.098182  0.098182       NaN   

                                                     
                         median       min       max  
infrastructure stage                                 
cloud-vm1      step    0.670788  0.000888  0.957272  
               step-1  0.007978  0.007978  0.007978  
               step-2  0.193996  0.193996  0.193996  
               step-3  0.097732  0.097732  0.097732  
local-lt1      step    0.055281  0.055281  0.055281  
local-lt2      step-1  0.000463  0.000463  0.000463  
               step-2  0.038446  0.038446  0.038446  
local-lt3      step-1  0.001496  0.001496  0.001496  
               step-2  0.098182  0.098182  0.098182

With these, we can start processing the collected results from Mlflow and Qdrant. In the former, open http://mlflow.oss:7001, click 'mlch-code-assistant-rag-pipeline' experiment and comparing the various results. In the latter, go to http://qdrant.oss:7001/dashboard, click collections, select 'internal_material_local_cloud_database' collection and check the vector points. We can gather the following details:

- RAG setup results (step-1):
    - Amount of points 447+
    - Used memory for the points is 3.8 MB
    - Used memory for sparse vectors is 1.8 MB
    - Used memory for dense vectors 1.3 mB
    - Processing time is 32 sec
    
- Data generator results (step-2):
    - VM1 results:
      - Prompt tokens:
            - Min is 457
            - Max is 3088
            - Mean is 1325
        - Completion tokens:
            - Min is 337
            - Max is 1321
            - Mean is 766
        - Total tokens:
            - Min is 951 
            - Max is 4018
            - Mean is 2092
        - Prompt processing time sec:
            - Min is 0.72
            - Max is 5.24
            - Mean is 2.17
        - Generation time sec:
            - Min is 11.12
            - Max is 51.18
            - Mean is 27.52
        - Total latency sec:
            - min is 12.42 
            - Max is 54.24
            - Mean is 29.69
        - Input tokens per sec:
            - Min is 527
            - Max is 670
            - Mean is 616
        - Output tokens per sec:
            - Min is 24.18
            - Max is 33.08
            - Mean is 28.69
        - Time to first token sec:
            - Min is 0.72
            - Max is 5.24
            - Mean is 2.17
        - Tokens per second:
            - Min is 19.69
            - Max is 31.63
            - Mean is 26.62
        - Time per output token sec:
            - Min is 0.03 
            - Max is o.041
            - Mean is 0.035
        - Input to output ratio:
            - Min is 0.58
            - Max is 5.88
            - Mean is 1.83
        - Context window utilization pct:
            - Min is 11.60
            - Max is 49.04
            - Mean is 25.53
    - LT2 results:
        - Prompt tokens:
            - Min is 377
            - Max is 3370
            - Mean is 1057
        - Completion tokens:
            - Min is 314
            - Max is 2208
            - Mean is 767
        - Total tokens:
            - Min is 697
            - Max is 4313
            - Mean is 1825
        - Prompt processing time sec:
            - Min is 0.1329
            - Max is 1.31
            - Mean is 0.40
        - Generation time sec:
            - Min is 6.90
            - Max is 52.66
            - Mean is 17.48
        - Total latency sec:
            - min is 7.21
            - Max is 52.84 
            - Mean is 17.89
        - Input tokens per sec:
            - Min is 1223
            - Max is 2880
            - Mean is 2591
        - Output tokens per sec:
            - Min is 40.57
            - Max is 46.82
            - Mean is 44.34
        - Time to first token sec:
            - Min is 0.132
            - Max is 1.31
            - Mean is 0.40
        - Tokens per second:
            - Min is 37.16
            - Max is 46.05
            - Mean is 43.18
        - Time per output token sec:
            - Min is 0.021 
            - Max is 0.024
            - Mean is 0.022
        - Input to output ratio:
            - Min is 0.22
            - Max is 8.776
            - Mean is 1.607
        - Context window utilization pct:
            - Min is 8.50
            - Max is 52.64
            - Mean is 22.28
    - LT3 results:
       - Prompt tokens:
            - Min is 346
            - Max is 2201
            - Mean is 740
        - Completion tokens:
            - Min is 265
            - Max is 2049
            - Mean is 714
        - Total tokens:
            - Min is 644
            - Max is 4026
            - Mean is 1454
        - Prompt processing time sec:
            - Min is 0.094
            - Max is 0.63
            - Mean is 0.209
        - Generation time sec:
            - Min is 4.18
            - Max is 35.21
            - Mean is 11.55
        - Total latency sec:
            - min is 4.28
            - Max is 35.78
            - Mean is 11.75
        - Input tokens per sec:
            - Min is 2941
            - Max is 3979
            - Mean is 3533
        - Output tokens per sec:
            - Min is 58.18
            - Max is 66.80
            - Mean is 62.21
        - Time to first token sec:
            - Min is 0.094
            - Max is 0.633
            - Mean is 0.209
        - Tokens per second:
            - Min is 55.65
            - Max is 64.90
            - Mean is 60.90
        - Time per output token sec:
            - Min is 0.014  
            - Max is 0.017
            - Mean is 0.016
        - Input to output ratio:
            - Min is 0.30
            - Max is 4.87
            - Mean is 1.23
        - Context window utilization pct:
            - Min is 7.86
            - Max is 49.14
            - Mean is 17.75
    - Processing time is 2143.24349 sec (35 min 43 sec)

- RAG evalution results (step-3):
    - Latency:
        - Embedding latency ms:
            - Mean is 2.70-17.61
            - Median is 2.70-17.61
            - P95 is 2.70-17.61
            - P99 is 2.70-16.51
        - Search latency ms:
            - Mean 8.66-29.99
            - Median 8.59-47.82
            - P95 is 9.9-59.02
            - P99 is 10-08-69.25
        - Total retrieval latency ms:
            - Mean is 12.81-36.62
            - Median is 12.75-50.53 
            - P95 is 14.06-73-53
            - P99 is 14.23-74-45
    - Characters:
        - Mean is 3525-8241 
        - Std is 1554-9350
        - Min is 861-3885
        - Max is 7065-77303
    - Ranking:
        - Precision@1:
          - Mean is 0.83-1.00
          - Std is 0-0.377
        - Recall@3:
          - Mean is 0.625-0.952
          - Std is 0.16-0.30
        - RR:
          - Mean is 0.906-1.00
          - Std is 0-0.2157
        - AP:
          - Mean is 0.696-0.877
          - Std is 0.181-0.255
        - NDCG@3:
          - Mean is 0.768-0.908
          - Std is 0.157-0.228
        - NDCG5:
          - Mean is 0.794-0.909
          - Std is 0.126-0.186
    - Processing time is 279.23551 sec (4 min 39 sec)

With these details, we can conclude the following actions for the model evalution stage:

- Coding assistant actions:
    - Generally put most inputs into LT3 when possible
    - LT2 might not be able to run the 9B models without specific config
- Assistant RAG actions:
    - The used dense and sparse models are quite fast with GPUs, but model loading might become a issue
    - Maybe the loading issue could be solved by running a RAG service in the VM1
    - Maybe cloud and local ray clusters need more slots to use GPUs
    - The models might require large context to enable 77303 cases or have formatting that keeps the context under the set limit
    - The search is pretty accurate on ideal material, but might face problems with relevance 1 material and inbalanced distributions
- Behavior controller actions:
    - LT2 might not be able to run alongside another model due to VRAM limits
    - Maybe VM1 should be made to run the behavior controller 
- General workflow actions:
    - Since we are going to make the different assistant variants answer general, synthetic and validation datasets that are collectively 370 QA pairs, the pipeline should handle each compared model as its own step, which mean the pipeline would have 5 steps with first 3 steps running the datasets against all assistant variants with the 2 last steps having LLM as a judge go through the datasets generated by the compared models
 
We now have done the necessery data preprocessing to proceed with model analysis stage.

---